# QQQ Fitting Pipeline

Loads hourly QQQ data, computes features and targets, then fits XGB model.

**Only end-of-day candles are used for prediction.**

**Features added:**
- Day of week
- Hourly lag returns over last 5 days (120 lag features)
- Technical indicators (RSI, EMA, volatility, etc.)

**Targets:**
- `T_overnight_perf`: overnight performance (close -> next open)
- `T_close_to_close`: close to next close performance

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from core.search_data import search_data
from norm.norm_utils import load_normalized_df
from features.base_dataframe import BaseDataFrame
from features.features_utils import FeatureType
from fitting.fitting_models import TimeSeriesModelTrainer, TrainingConfig
from fitting.fitting_core import TrainingSplitType, TaskType, RetrainMode
from fitting.models import ModelFactory, ModelType
from core.enums import g_close_col, g_open_col, g_index_col
from strat.strat_filters import trailing_stop_numba
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

MINUTES_PER_DAY = 1440
HOURS_PER_DAY = 24

/usr/local/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## 1. Load QQQ Data

In [3]:
files = search_data(p_symbol="QQQ", p_data_freq="candle_1hour")
print(f"Found {len(files)} file(s)")
for f in files:
    print(f"  {f.path}")

Found 1 file(s)
  /app/data/normalised/candle_1hour/firstrate_undefined/etf/QQQ/QQQ_*_*_candle_1hour.df.parquet


In [4]:
df = load_normalized_df(str(files[0].path))
print(f"Loaded {len(df)} rows")
df.head()

Loaded 92274 rows


,i_minute_i,S_open_f32,S_high_f32,S_low_f32,S_close_f32,S_volume_f64,S_open_time_i,S_close_time_i
0,3420,84.791100,84.791100,82.091499,82.174103,8984264.0,2000-01-03 09:00:00,2000-01-03 09:00:00
1,3480,82.201698,83.165802,80.218300,80.824303,9491568.0,2000-01-03 10:00:00,2000-01-03 10:00:00
2,3540,80.851799,82.119003,79.997902,81.898697,8086721.0,2000-01-03 11:00:00,2000-01-03 11:00:00
3,3600,81.871101,81.981300,81.209999,81.664497,2276071.0,2000-01-03 12:00:00,2000-01-03 12:00:00
4,3660,81.719597,82.642403,81.485397,82.311897,3187450.0,2000-01-03 13:00:00,2000-01-03 13:00:00


## 2. Compute Technical Features

In [5]:
bdf = BaseDataFrame(p_df=df, p_verbose=False)

bdf.add_feature(FeatureType.RSI, periods=[14, 60, 240])
bdf.add_feature(FeatureType.SPREAD_REL_EMA, periods=[15, 60, 240])
bdf.add_feature(FeatureType.DIFF_REL_EMA_MID, periods=[15, 60, 240])
bdf.add_feature(FeatureType.HIST_VOLATILITY, periods=[15, 60, 240])
bdf.add_feature(FeatureType.ROC, periods=[14, 60])
bdf.add_feature(FeatureType.ADI)

df = bdf.get_dataframe()
feature_cols = bdf.get_feature_columns()

print(f"Generated {len(feature_cols)} technical features")

Generated 17 technical features


## 3. Add Day Features (Day of Week + 5 Days of Hourly Lag Returns)

In [6]:
base = pd.Timestamp("2000-01-01")
df["datetime"] = base + pd.to_timedelta(df[g_index_col], unit="m")

df["F_day_of_week"] = df["datetime"].dt.dayofweek.astype(np.float32)
day_feature_cols = ["F_day_of_week"]

#for lag_hours in range(1, 5 * HOURS_PER_DAY + 1):
#    col_name = f"F_lag_ret_{lag_hours}h_f16"
#    df[col_name] = (df[g_close_col] - df[g_close_col].shift(lag_hours)) / df[g_close_col].shift(lag_hours)
#    df[col_name] = df[col_name].replace([np.inf, -np.inf], np.nan)
#    day_feature_cols.append(col_name)

feature_cols = feature_cols + day_feature_cols
print(f"Added {len(day_feature_cols)} day features (1 day_of_week + 120 hourly lag returns)")
print(f"Total features: {len(feature_cols)}")

Added 1 day features (1 day_of_week + 120 hourly lag returns)
Total features: 18


In [7]:
df[day_feature_cols].describe().T.head(15)

,count,mean,std,min,25%,50%,75%,max
F_day_of_week,92274.0,2.018694,1.399139,0.0,1.0,2.0,3.0,4.0


## 4. Compute Targets

In [8]:
df["T_overnight_perf"] = (df[g_open_col].shift(-1) - df[g_close_col]) / df[g_close_col]
df["T_close_to_close"] = (df[g_close_col].shift(-1) - df[g_close_col]) / df[g_close_col]

df["T_overnight_perf"] = df["T_overnight_perf"].replace([np.inf, -np.inf], np.nan)
df["T_close_to_close"] = df["T_close_to_close"].replace([np.inf, -np.inf], np.nan)

target_cols = ["T_overnight_perf", "T_close_to_close"]
print(f"Generated {len(target_cols)} targets")

Generated 2 targets


In [9]:
df[target_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
T_overnight_perf,92273.0,0.000021,0.001915,-0.082771,-0.000090,0.000000,0.000100,0.06411
T_close_to_close,92273.0,0.000031,0.004430,-0.071444,-0.001202,0.000059,0.001408,0.09838


## 5. Detect End-of-Day Candles

In [10]:
df["day_num"] = df[g_index_col] // MINUTES_PER_DAY
df["next_day_num"] = df["day_num"].shift(-1)
df["is_eod"] = df["day_num"] != df["next_day_num"]
df.loc[df["is_eod"].isna(), "is_eod"] = False

print(f"Found {df['is_eod'].sum()} end-of-day candles out of {len(df)} total")
print(f"~{df['is_eod'].sum() / (len(df) / 24):.0f} trading days")

Found 6577 end-of-day candles out of 92274 total
~2 trading days


In [11]:
df[df["is_eod"]][[g_index_col, "day_num", g_close_col, "F_day_of_week", "T_overnight_perf", "T_close_to_close"]].head(10)

,i_minute_i,day_num,S_close_f32,F_day_of_week,T_overnight_perf,T_close_to_close
7,3840,2,98.379272,0.0,-0.029024,-0.017481
17,5400,3,91.630272,1.0,-0.008499,-0.033995
27,6840,4,89.294113,2.0,0.010174,0.002179
37,8280,5,83.161606,3.0,0.035505,0.047210
47,9720,6,93.447235,4.0,0.011112,0.016667
57,14040,9,96.042992,0.0,-0.008108,0.001690
67,15480,10,91.370674,1.0,0.011363,-0.012784
76,16920,11,89.359009,2.0,0.028322,0.023601
86,18360,12,94.745110,3.0,0.019179,0.013699
95,19800,13,96.951591,4.0,-0.014056,-0.009538


## 6. Prepare Training Data

In [12]:
def prepare_data(df, feature_cols, target_col, eod_only=True):
    valid_mask = ~df[feature_cols].isna().any(axis=1)
    valid_mask &= ~df[target_col].isna()
    
    if eod_only:
        valid_mask &= df["is_eod"]
    
    X = df.loc[valid_mask, feature_cols].values.astype(np.float32)
    y = df.loc[valid_mask, target_col].values.astype(np.float32)
    
    print(f"Prepared {X.shape[0]} samples with {X.shape[1]} features (eod_only={eod_only})")
    return X, y

## 7. Fit XGB Model

In [13]:
def fit_model(X, y, n_estimators=200, max_depth=6, learning_rate=0.1,
              train_ratio=0.2, sliding_window_size=126, retrain_period=21):
    config = TrainingConfig(
        mode=TrainingSplitType.PERIODIC_RETRAIN,
        train_ratio=train_ratio,
        retrain_period=retrain_period,
        retrain_mode=RetrainMode.SLIDING,
        sliding_window_size=sliding_window_size,
        normalization="standardize",
    )
    
    model = ModelFactory.create_model(
        model_type=ModelType.XGB,
        task_type=TaskType.REGRESSION,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
    )
    
    trainer = TimeSeriesModelTrainer(
        model=model,
        config=config,
        task_type=TaskType.REGRESSION,
        verbose=False,
    )
    
    metrics = trainer.fit(X, y)
    return trainer, metrics

### Target: Overnight Performance

In [14]:
target_col = "T_overnight_perf"
X, y = prepare_data(df, feature_cols, target_col, eod_only=True)

Prepared 6551 samples with 18 features (eod_only=True)


In [15]:
trainer_overnight, metrics_overnight = fit_model(X, y)

print(f"\nResults for {target_col}:")
print(f"  Train RMSE: {metrics_overnight.train_score:.6f}")
if metrics_overnight.val_score is not None:
    print(f"  Val RMSE:   {metrics_overnight.val_score:.6f}")
print(f"  Test RMSE:  {metrics_overnight.test_score:.6f}")


Results for T_overnight_perf:
  Train RMSE: 0.000000
  Test RMSE:  0.000000


In [16]:
print("Train metrics:", metrics_overnight.train_metrics)
print("Val metrics:", metrics_overnight.val_metrics)
print("Test metrics:", metrics_overnight.test_metrics)

Train metrics: {'mse': 0.000126933651042422, 'rmse': 0.011266483526035176, 'mae': 0.007545799914863656, 'r2': -0.12011926642971749, 'score': 0.0}
Val metrics: None
Test metrics: {'mse': 3.7056089676708055e-05, 'rmse': 0.006087371327322497, 'mae': 0.0036646213479824465, 'r2': -0.397341688422469, 'score': 0.0}


### Target: Close to Close

In [17]:
target_col = "T_close_to_close"
X, y = prepare_data(df, feature_cols, target_col, eod_only=True)

Prepared 6551 samples with 18 features (eod_only=True)


In [24]:
trainer_ctc, metrics_ctc = fit_model(X, y)

print("Train metrics:", metrics_ctc.train_metrics)
print("Val metrics:", metrics_ctc.val_metrics)
print("Test metrics:", metrics_ctc.test_metrics)

Train metrics: {'mse': 0.0001541359590539185, 'rmse': 0.012415150383862391, 'mae': 0.008899759154774907, 'r2': -0.11693064521264063, 'score': 0.0}
Val metrics: None
Test metrics: {'mse': 4.8532087376428706e-05, 'rmse': 0.006966497497051779, 'mae': 0.004483729804207608, 'r2': -0.3379624236376997, 'score': 0.0}


In [22]:
from strat.strat_filters import trailing_stop_numba

USE_trailing_stop = True
atr_mult = 2.0

print(f"Using trailing stop with atr_mult={atr_mult}")

#print(f"Trailing stop applied: Mean alloc: {ts_mean():.4f}, std:: {ts.mean():.4f}, std: {ts.std():.4f}")

#print(f"Trailing stop signal: non-zero: {ts.sum()}")

Using trailing stop with atr_mult=2.0


NameError: name 'metrics_ctc' is not defined

## 8. Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

target_col = "T_close_to_close"
X_test, y_test = prepare_data(df, feature_cols, target_col, eod_only=True)

preds = trainer_ctc.predict(X_test[-500:])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(y_test[-500:], label='Actual', alpha=0.7)
axes[0].plot(preds, label='Predicted', alpha=0.7)
axes[0].set_title(f'{target_col}: Actual vs Predicted (last 500 EOD samples)')
axes[0].legend()
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Return')

axes[1].scatter(y_test[-500:], preds, alpha=0.3, s=10)
axes[1].plot([-0.05, 0.05], [-0.05, 0.05], 'r--', label='Perfect prediction')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Predicted vs Actual Scatter')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
importance = trainer_ctc.model._model.feature_importances_

feat_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance
}).sort_values('importance', ascending=False)

feat_importance.head(20).plot.barh(x='feature', y='importance', figsize=(10, 8), legend=False)
plt.title('Top 20 Feature Importances (Close-to-Close)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
feat_importance.head(30)

## 10. Check Day of Week Feature Importance

In [ ]:
feat_importance[feat_importance['feature'].isin(['F_day_of_week'] + [f'F_lag_ret_{h}h_f16' for h in [1, 2, 3, 4, 5, 24, 48, 72, 96, 120]])]

## 11. Compare: EOD-only vs All Candles

In [ ]:
target_col = "T_close_to_close"

print("="*60)
print("EOD-only training:")
X_eod, y_eod = prepare_data(df, feature_cols, target_col, eod_only=True)
_, metrics_eod = fit_model(X_eod, y_eod)
print(f"  Train RMSE: {metrics_eod.train_score:.6f}")
print(f"  Test RMSE:  {metrics_eod.test_score:.6f}")

print("\n" + "="*60)
print("All candles training:")
X_all, y_all = prepare_data(df, feature_cols, target_col, eod_only=False)
_, metrics_all = fit_model(X_all, y_all)
print(f"  Train RMSE: {metrics_all.train_score:.6f}")
print(f"  Test RMSE:  {metrics_all.test_score:.6f}")

## 12. Dual-Target Strategy: Overnight + Close-to-Close

### Strategy Logic:
- Train both overnight and close-to-close models
- Go **LONG** if both models predict positive return
- Go **SHORT** if both models predict negative return
- Enter at EOD close, exit at midday (12:00) next day

In [ ]:
from backtest.backtest_basket_alloc_based import run_full_backtest
from backtest.backtest_utils import compute_strategy_metrics

In [ ]:
target_overnight = "T_overnight_perf"
X_on, y_on = prepare_data(df, feature_cols, target_overnight, eod_only=True)
trainer_on, metrics_on = fit_model(X_on, y_on, train_ratio=0.2)

print(f"\nOvernight Model - Train RMSE: {metrics_on.train_score:.6f}, Test RMSE: {metrics_on.test_score:.6f}")

In [ ]:
target_ctc = "T_close_to_close"
X_ctc, y_ctc = prepare_data(df, feature_cols, target_ctc, eod_only=True)
trainer_ctc, metrics_ctc = fit_model(X_ctc, y_ctc, train_ratio=0.2)

print(f"\nClose-to-Close Model - Train RMSE: {metrics_ctc.train_score:.6f}, Test RMSE: {metrics_ctc.test_score:.6f}")

In [ ]:
print("\n=== Per-Chunk Performance (Overnight Model) ===")
if metrics_on.retrain_history:
    chunk_df_on = pd.DataFrame(metrics_on.retrain_history)
    chunk_df_on['test_r2'] = chunk_df_on['test_chunk_metrics'].apply(lambda x: x.get('r2', 0) if x else 0)
    chunk_df_on['test_rmse'] = chunk_df_on['test_chunk_metrics'].apply(lambda x: x.get('rmse', 0) if x else 0)
    print(chunk_df_on[['index', 'train_size', 'test_size', 'test_r2', 'test_rmse']].head(15))
else:
    print("No retrain history (static training mode)")

In [ ]:
print("\n=== Per-Chunk Performance (Close-to-Close Model) ===")
if metrics_ctc.retrain_history:
    chunk_df_ctc = pd.DataFrame(metrics_ctc.retrain_history)
    chunk_df_ctc['test_r2'] = chunk_df_ctc['test_chunk_metrics'].apply(lambda x: x.get('r2', 0) if x else 0)
    chunk_df_ctc['test_rmse'] = chunk_df_ctc['test_chunk_metrics'].apply(lambda x: x.get('rmse', 0) if x else 0)
    print(chunk_df_ctc[['index', 'train_size', 'test_size', 'test_r2', 'test_rmse']].head(15))
else:
    print("No retrain history (static training mode)")

In [ ]:
pred_overnight = trainer_on.predict(X_on)
pred_ctc = trainer_ctc.predict(X_ctc)

print(f"Overnight predictions: mean={pred_overnight.mean():.6f}, std={pred_overnight.std():.6f}")
print(f"CTC predictions: mean={pred_ctc.mean():.6f}, std={pred_ctc.std():.6f}")

In [ ]:
combined_signal = np.zeros(len(pred_overnight))
combined_signal[(pred_overnight > 0) & (pred_ctc > 0)] = 1.0   # Long
combined_signal[(pred_overnight < 0) & (pred_ctc < 0)] = -1.0  # Short

n_long = (combined_signal == 1).sum()
n_short = (combined_signal == -1).sum()
n_flat = (combined_signal == 0).sum()

print(f"Signal distribution: Long={n_long}, Short={n_short}, Flat={n_flat}")

In [ ]:
valid_mask = ~df[feature_cols].isna().any(axis=1)
valid_mask &= ~df['T_overnight_perf'].isna()
valid_mask &= ~df['T_close_to_close'].isna()
valid_mask &= df['is_eod']

eod_indices = np.where(valid_mask)[0]

eod_df = df[valid_mask].copy()
eod_df = eod_df.reset_index(drop=True)
eod_df['pred_overnight'] = pred_overnight
eod_df['pred_ctc'] = pred_ctc
eod_df['signal'] = combined_signal
eod_df['actual_overnight'] = eod_df['T_overnight_perf'].values
eod_df['actual_ctc'] = eod_df['T_close_to_close'].values

print(f"EOD samples: {len(eod_df)}")
eod_df[['datetime', 'signal', 'pred_overnight', 'pred_ctc', 'actual_overnight', 'actual_ctc']].head(10)

In [ ]:
alloc_df = df[[g_index_col, g_close_col, 'day_num']].copy()
alloc_df['A_QQQ_alloc'] = 0.0

signal_idx = 0
for idx in eod_indices:
    if signal_idx < len(combined_signal):
        alloc_df.loc[idx, 'A_QQQ_alloc'] = combined_signal[signal_idx]
        signal_idx += 1

print(f"Allocation entries set: {(alloc_df['A_QQQ_alloc'] != 0).sum()}")

In [ ]:
exit_positions = {}
for idx in eod_indices:
    if idx < len(alloc_df):
        current_alloc = alloc_df.loc[idx, 'A_QQQ_alloc']
        if current_alloc != 0:
            day_num = alloc_df.loc[idx, 'day_num']
            midday_minute = int((day_num + 1) * MINUTES_PER_DAY + 12 * 60)
            exit_positions[midday_minute] = current_alloc

for midday_minute, prev_alloc in exit_positions.items():
    mask = alloc_df[g_index_col] == midday_minute
    if mask.any():
        alloc_df.loc[mask, 'A_QQQ_alloc'] = 0.0

print(f"Exit positions set at midday: {len(exit_positions)}")

In [ ]:
backtest_df = df[[g_index_col, g_close_col]].copy()
backtest_df['QQQ_S_close_f32'] = backtest_df[g_close_col]

alloc_for_merge = alloc_df[[g_index_col, 'A_QQQ_alloc']].copy()
backtest_df = backtest_df.merge(alloc_for_merge, on=g_index_col, how='left')
backtest_df['A_QQQ_alloc'] = backtest_df['A_QQQ_alloc'].fillna(0)

backtest_df = backtest_df.set_index(g_index_col)

print(f"Backtest dataframe: {len(backtest_df)} rows")
print(f"Non-zero allocations: {(backtest_df['A_QQQ_alloc'] != 0).sum()}")

In [ ]:
result_df, orders_df = run_full_backtest(
    p_df=backtest_df,
    p_asset_list=['QQQ'],
    transaction_cost_pct=0.001
)

print(f"\nOrders executed: {len(orders_df)}")
if len(orders_df) > 0:
    print(orders_df.head(20))

In [ ]:
metrics = compute_strategy_metrics(result_df, orders_df, ['QQQ'])

print("\n" + "="*60)
print("STRATEGY PERFORMANCE METRICS")
print("="*60)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(result_df['port_value'].values)
axes[0].set_title('Portfolio Value Over Time')
axes[0].set_xlabel('Bar')
axes[0].set_ylabel('Portfolio Value')
axes[0].grid(True, alpha=0.3)

axes[1].plot(result_df['cum_pnl_pct'].values * 100)
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_title('Cumulative PnL (%)')
axes[1].set_xlabel('Bar')
axes[1].set_ylabel('PnL (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [19]:
long_returns = eod_df[eod_df['signal'] == 1]['actual_overnight']
short_returns = eod_df[eod_df['signal'] == -1]['actual_overnight']

print("\n=== Signal Performance Analysis ===")
print(f"\nLONG signals (n={len(long_returns)}):")
print(f"  Mean overnight return: {long_returns.mean():.6f}")
print(f"  Win rate: {(long_returns > 0).mean():.2%}")

print(f"\nSHORT signals (n={len(short_returns)}):")
print(f"  Mean overnight return: {short_returns.mean():.6f}")
print(f"  Win rate: {(short_returns < 0).mean():.2%}")

all_signal_returns = pd.concat([long_returns, -short_returns])
print(f"\nCombined (long wins + short wins):")
print(f"  Mean return: {all_signal_returns.mean():.6f}")
print(f"  Win rate: {(all_signal_returns > 0).mean():.2%}")

NameError: name 'eod_df' is not defined